In [ ]:
import torch
import torch.autograd.functional as F
import torch.distributions as dist

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

# Retina display
%config InlineBackend.figure_format = 'retina'

In [ ]:
from tueplots import bundles
plt.rcParams.update(bundles.beamer_moml())
#plt.rcParams.update(bundles.icml2022())


# Also add despine to the bundle using rcParams
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False

# Increase font size to match Beamer template
plt.rcParams['font.size'] = 16
# Make background transparent
plt.rcParams['figure.facecolor'] = 'none'


### Parameters

In [ ]:
# Prior distribution
P = torch.tensor([0.5, 0.5])

# Transition matrix
A = torch.tensor([[0.9, 0.1],
                  [0.2, 0.8]])

# States 
STATES = {0: 'Sunny', 1: 'Rainy'}

### Sampling from discrete state Markov chain

In [ ]:
# Generate data from the Markov chain

def generate_data(n_samples=10, seed=0):
    torch.manual_seed(seed)
    data = torch.zeros(n_samples, dtype=torch.long)
    data[0] = dist.Categorical(P).sample()
    for i in range(1, n_samples):
        data[i] = dist.Categorical(A[data[i-1]]).sample()
    return data

def map_state(x_arr):
    return [STATES[x.item()] for x in x_arr]

In [ ]:
generate_data(10, 0)
map_state(generate_data(10, 0))

In [ ]:
map_state(generate_data(10, 1))

### Stationary distribution

In [ ]:
PS = []
PR = []

In [ ]:
# P(Sunny) at t=0
PS.append(P[0].item())
PS

In [ ]:
# P (Rainy) at t=0
PR.append(P[1].item())
PR

In [ ]:
def PS_t(t):
    return PS[t-1] * A[0, 0].item() + PR[t-1] * A[1, 0].item()

def PR_t(t):
    return PS[t-1] * A[0, 1].item() + PR[t-1] * A[1, 1].item()


In [ ]:
for t in range(1, 20):
    PS.append(PS_t(t))
    PR.append(PR_t(t))


In [ ]:
PS, PR

In [ ]:
plt.plot(PS, label='P(Sunny)')
plt.plot(PR, label='P(Rainy)')
plt.xlabel('Time')
plt.ylabel('Probability')
plt.legend()

In [ ]:
A@A, torch.matrix_power(A, 2)

#### Vectorised version

In [ ]:
# Distribution of states at t = 0
P

In [ ]:
# P(Sunny) at t = 1
print("--"*20)
print("P(Sunny) at t = 1")
print(P[0] * A[0, 0] + P[1] * A[1, 0])

# P(Rainy) at t = 1
print("--"*20)
print("P(Rain) at t = 1")
print(P[0] * A[0, 1] + P[1] * A[1, 1])

# Distribution of states at t = 1
print("--"*20)
print("Distribution of states at t = 1")
print(P@A)

In [ ]:
# Distribution of states at t = 2
print("--"*20)
print("Distribution of states at t = 2")
print(P@A@A)

# Using torch.matrix_power
print("--"*20)
print("Distribution of states at t = 2")
print(P@torch.matrix_power(A, 2))

In [ ]:
# convergence
P@torch.matrix_power(A, 100), P@torch.matrix_power(A, 99)

In [ ]:
PS

In [ ]:
### What if we started with different initial distribution?

P2 = torch.tensor([0.1, 0.9])
P3 = torch.tensor([0.9, 0.1])
P4 = torch.tensor([0.999999, 0.000001])

P2@torch.matrix_power(A, 100), P3@torch.matrix_power(A, 100), P4@torch.matrix_power(A, 100)

In [ ]:
### Checking for convergence of Markov chain using iterative method

eps = 1e-6
PS = P
for i in range(100):
    PS_new = PS@A
    if torch.all(torch.abs(PS_new - PS) < eps):
        print("Converged at iteration", i)
        break
    PS = PS_new

In [ ]:
### Checking for convergence

def check_convergence(P, A, iter=100, eps=1e-6):
    for i in range(iter):
        P_new = P@A
        if torch.all(torch.abs(P_new - P) < eps):
            print("Converged at iteration", i)
            break
        P = P_new
    return P

In [ ]:
check_convergence(P, A)

In [ ]:
check_convergence(P2, A)

In [ ]:
check_convergence(P3, A)

In [ ]:
check_convergence(P4, A)

### Homogeneous Markov chain

The transition matrix is the same for all time steps.


### Irreducible Markov chain

A Markov chain is irreducible if it is possible to get to any state from any state.

### Aperiodic Markov chain

A Markov chain is aperiodic if there are no cycles in the state transition graph.


In [ ]:
### Markov chain with aperiodic transition matrix

A = torch.tensor([[0.9, 0.1],
                    [0.2, 0.8]])






In [ ]:
### Continuous space Markov chain

def transition(a, b):
    return dist.Normal(a, 0.1).log_prob(b).exp()


